In [2]:
from pathlib import Path
import json
from datetime import datetime, timezone

import pandas as pd
import geopandas as gpd
import rasterio


# --- Project paths ---
ROOT = Path(r"C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton")
DATA = ROOT / "data_raw"
OUT  = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

# --- Input files ---
adm2_path = DATA / "ZMB_adm2_gadm41_Copperbelt.shp"
pop_path  = DATA / "zmb_ppp_2020_constrained.tif"
smod_path = DATA / "GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V1_0_R11_C21.tif"

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")

# ---------- Helpers ----------
def raster_meta(path: Path) -> dict:
    with rasterio.open(path) as src:
        return {
            "path": str(path),
            "crs": str(src.crs),
            "width": int(src.width),
            "height": int(src.height),
            "bands": int(src.count),
            "dtype": list(src.dtypes),
            "nodata": src.nodata,
            "pixel_size": tuple(float(x) for x in src.res),
            "bounds": {
                "left": float(src.bounds.left),
                "bottom": float(src.bounds.bottom),
                "right": float(src.bounds.right),
                "top": float(src.bounds.top),
            },
            "driver": src.driver,
        }

def vector_meta(path: Path) -> dict:
    gdf = gpd.read_file(path)
    b = gdf.total_bounds  # [minx, miny, maxx, maxy]
    meta = {
        "path": str(path),
        "crs": str(gdf.crs),
        "feature_count": int(len(gdf)),
        "geometry_type": str(gdf.geom_type.iloc[0]) if len(gdf) else None,
        "columns": list(gdf.columns),
        "bounds": {"minx": float(b[0]), "miny": float(b[1]), "maxx": float(b[2]), "maxy": float(b[3])},
    }
    if "NAME_2" in gdf.columns:
        meta["NAME_2_sample"] = gdf["NAME_2"].head(10).tolist()
    return meta

# ---------- Build metadata record ----------
record = {
    "run_time_utc": run_time_utc,
    "project_root": str(ROOT),
    "folders": {
        "root_exists": ROOT.exists(),
        "data_raw_exists": DATA.exists(),
        "outputs_exists": OUT.exists(),
    },
    "files_exist": {
        "adm2": adm2_path.exists(),
        "worldpop": pop_path.exists(),
        "smod": smod_path.exists(),
    },
    "layers": {
        "adm2": vector_meta(adm2_path),
        "worldpop": raster_meta(pop_path),
        "smod": raster_meta(smod_path),
    }
}

# ---------- Print (quick human check) ----------
print("Run time (UTC):", record["run_time_utc"])
print("ROOT exists:", record["folders"]["root_exists"])
print("DATA exists:", record["folders"]["data_raw_exists"])
print("OUT exists :", record["folders"]["outputs_exists"])
print("\nFiles exist:", record["files_exist"])

print("\nADM2:", record["layers"]["adm2"]["feature_count"], "features | CRS:", record["layers"]["adm2"]["crs"])
print("WorldPop: CRS:", record["layers"]["worldpop"]["crs"],
      "| dims:", (record["layers"]["worldpop"]["width"], record["layers"]["worldpop"]["height"]),
      "| res:", record["layers"]["worldpop"]["pixel_size"],
      "| nodata:", record["layers"]["worldpop"]["nodata"])
print("SMOD: CRS:", record["layers"]["smod"]["crs"],
      "| dims:", (record["layers"]["smod"]["width"], record["layers"]["smod"]["height"]),
      "| res:", record["layers"]["smod"]["pixel_size"],
      "| nodata:", record["layers"]["smod"]["nodata"])

# ---------- Save JSON artefact ----------
json_path = OUT / "00_sanity_check_metadata.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(record, f, indent=2)

# ---------- Save a tidy CSV summary ----------
rows = []
for layer_name, meta in record["layers"].items():
    if layer_name == "adm2":
        rows.append({
            "layer": "adm2",
            "type": "vector",
            "crs": meta["crs"],
            "feature_count": meta["feature_count"],
            "width": None,
            "height": None,
            "pixel_x": None,
            "pixel_y": None,
            "bands": None,
            "dtype": None,
            "nodata": None,
            "bounds": f'{meta["bounds"]["minx"]},{meta["bounds"]["miny"]},{meta["bounds"]["maxx"]},{meta["bounds"]["maxy"]}',
            "path": meta["path"],
        })
    else:
        rows.append({
            "layer": layer_name,
            "type": "raster",
            "crs": meta["crs"],
            "feature_count": None,
            "width": meta["width"],
            "height": meta["height"],
            "pixel_x": meta["pixel_size"][0],
            "pixel_y": meta["pixel_size"][1],
            "bands": meta["bands"],
            "dtype": ";".join(meta["dtype"]),
            "nodata": meta["nodata"],
            "bounds": f'{meta["bounds"]["left"]},{meta["bounds"]["bottom"]},{meta["bounds"]["right"]},{meta["bounds"]["top"]}',
            "path": meta["path"],
        })

csv_path = OUT / "00_sanity_check_layers.csv"
pd.DataFrame(rows).to_csv(csv_path, index=False)

print("\nSaved:")
print(" -", json_path)
print(" -", csv_path)


Run time (UTC): 2026-01-24T20:31:52+00:00
ROOT exists: True
DATA exists: True
OUT exists : True

Files exist: {'adm2': True, 'worldpop': True, 'smod': True}

ADM2: 12 features | CRS: EPSG:4326
WorldPop: CRS: EPSG:4326 | dims: (14047, 11826) | res: (0.0008333333300348827, 0.000833333330035515) | nodata: -99999.0
SMOD: CRS: ESRI:54009 | dims: (1000, 1000) | res: (1000.0, 1000.0) | nodata: -200.0

Saved:
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\00_sanity_check_metadata.json
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\00_sanity_check_layers.csv
